In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🛡️ Kubernetes Advanced Patterns Guide

**Advanced Kubernetes operators, custom controllers, and platform engineering patterns**

This guide covers:
- Custom resource definitions (CRDs)
- Operators pattern
- Custom controllers
- KEDA for autoscaling
- Service mesh integration
- Policy enforcement
- Resource management
- Observability in Kubernetes
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Custom Resource Definitions & Operators

### CRD Manager
```python
from dataclasses import dataclass, field
from typing import Dict, Optional, List
from datetime import datetime
from enum import Enum

class ResourcePhase(Enum):
    PENDING = "Pending"
    ACTIVE = "Active"
    DELETING = "Deleting"
    DELETED = "Deleted"

@dataclass
class CRDDefinition:
    """Custom Resource Definition"""
    group: str
    name: str
    kind: str
    schema: Dict
    version: str
    scope: str = "Namespaced"  # Namespaced or Cluster

@dataclass
class CustomResource:
    """Custom Resource instance"""
    namespace: str
    name: str
    kind: str
    spec: Dict
    status: Dict = field(default_factory=dict)
    metadata: Dict = field(default_factory=dict)
    phase: ResourcePhase = ResourcePhase.PENDING
    created_at: datetime = field(default_factory=datetime.utcnow)
    uid: str = ""

class CRDController:
    """Manage custom resources"""
    
    def __init__(self):
        self.crds: Dict[str, CRDDefinition] = {}
        self.resources: Dict[str, Dict[str, CustomResource]] = {}
        self.watchers: List[callable] = []
    
    def register_crd(self, crd: CRDDefinition) -> bool:
        """Register CRD"""
        
        crd_key = f"{crd.group}/{crd.kind}"
        self.crds[crd_key] = crd
        self.resources[crd_key] = {}
        
        return True
    
    def create_resource(self, crd_key: str, resource: CustomResource) -> bool:
        """Create custom resource"""
        
        if crd_key not in self.crds:
            return False
        
        resource_key = f"{resource.namespace}/{resource.name}"
        self.resources[crd_key][resource_key] = resource
        resource.phase = ResourcePhase.ACTIVE
        
        # Notify watchers
        self._notify_watchers('ADDED', resource)
        
        return True
    
    def get_resource(self, crd_key: str, namespace: str, name: str) -> Optional[CustomResource]:
        """Get custom resource"""
        
        if crd_key not in self.resources:
            return None
        
        resource_key = f"{namespace}/{name}"
        return self.resources[crd_key].get(resource_key)
    
    def update_resource(self, crd_key: str, resource: CustomResource) -> bool:
        """Update custom resource"""
        
        if crd_key not in self.resources:
            return False
        
        resource_key = f"{resource.namespace}/{resource.name}"
        existing = self.resources[crd_key].get(resource_key)
        
        if not existing:
            return False
        
        self.resources[crd_key][resource_key] = resource
        
        # Notify watchers
        self._notify_watchers('MODIFIED', resource)
        
        return True
    
    def delete_resource(self, crd_key: str, namespace: str, name: str) -> bool:
        """Delete custom resource"""
        
        if crd_key not in self.resources:
            return False
        
        resource_key = f"{namespace}/{name}"
        resource = self.resources[crd_key].get(resource_key)
        
        if not resource:
            return False
        
        resource.phase = ResourcePhase.DELETING
        
        # Notify watchers
        self._notify_watchers('DELETED', resource)
        
        del self.resources[crd_key][resource_key]
        
        return True
    
    def watch_resources(self, crd_key: str, callback: callable):
        """Watch for resource changes"""
        
        self.watchers.append((crd_key, callback))
    
    def _notify_watchers(self, event_type: str, resource: CustomResource):
        """Notify watchers"""
        
        for crd_key, callback in self.watchers:
            callback({
                'type': event_type,
                'object': resource
            })

class OperatorController:
    """Base operator controller"""
    
    def __init__(self, crd_controller: CRDController, crd_key: str):
        self.crd_controller = crd_controller
        self.crd_key = crd_key
        self.reconciliation_queue: List[str] = []
    
    def reconcile(self, namespace: str, name: str) -> bool:
        """Reconcile desired state with actual state"""
        
        resource = self.crd_controller.get_resource(self.crd_key, namespace, name)
        
        if not resource:
            return False
        
        # Get desired state
        desired_state = resource.spec
        
        # Get current state
        current_state = resource.status
        
        # Compare and update if needed
        if desired_state != current_state:
            # Apply desired state
            resource.status = desired_state
            resource.status['synced'] = True
            
            self.crd_controller.update_resource(self.crd_key, resource)
        
        return True
    
    def enqueue_reconciliation(self, namespace: str, name: str):
        """Enqueue reconciliation task"""
        
        key = f"{namespace}/{name}"
        if key not in self.reconciliation_queue:
            self.reconciliation_queue.append(key)
    
    def process_queue(self) -> int:
        """Process reconciliation queue"""
        
        processed = 0
        
        while self.reconciliation_queue:
            key = self.reconciliation_queue.pop(0)
            namespace, name = key.split('/')
            
            if self.reconcile(namespace, name):
                processed += 1
        
        return processed
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. KEDA & Autoscaling

### KEDA Scaler
```python
@dataclass
class ScalerConfig:
    """KEDA scaler configuration"""
    trigger_type: str  # cpu, memory, queue, webhook, etc.
    min_replicas: int
    max_replicas: int
    target_value: float

class KEDAScaler:
    """KEDA autoscaler"""
    
    def __init__(self):
        self.scalers: Dict[str, ScalerConfig] = {}
        self.current_replicas: Dict[str, int] = {}
        self.metrics_history: Dict[str, List[float]] = {}
    
    def configure_scaler(self, resource_name: str, config: ScalerConfig) -> bool:
        """Configure KEDA scaler"""
        
        self.scalers[resource_name] = config
        self.current_replicas[resource_name] = config.min_replicas
        self.metrics_history[resource_name] = []
        
        return True
    
    def evaluate_scaling(self, resource_name: str, current_metric: float) -> Optional[int]:
        """Evaluate if scaling needed"""
        
        if resource_name not in self.scalers:
            return None
        
        config = self.scalers[resource_name]
        
        # Record metric
        self.metrics_history[resource_name].append(current_metric)
        
        # Calculate target replicas
        if config.trigger_type == 'cpu':
            target = self._calculate_replicas_from_cpu(current_metric, config)
        elif config.trigger_type == 'queue':
            target = self._calculate_replicas_from_queue(current_metric, config)
        else:
            return None
        
        # Enforce min/max
        target = max(config.min_replicas, min(config.max_replicas, target))
        
        # Scale if different
        if target != self.current_replicas[resource_name]:
            self.current_replicas[resource_name] = target
            return target
        
        return None
    
    def _calculate_replicas_from_cpu(self, cpu_percent: float, config: ScalerConfig) -> int:
        """Calculate replicas from CPU metric"""
        
        if cpu_percent <= config.target_value * 0.7:
            return config.min_replicas
        
        if cpu_percent >= config.target_value * 1.3:
            return config.max_replicas
        
        # Linear scaling between min and max
        scale_factor = (cpu_percent - config.target_value * 0.7) / (config.target_value * 0.6)
        target = config.min_replicas + int((config.max_replicas - config.min_replicas) * scale_factor)
        
        return target
    
    def _calculate_replicas_from_queue(self, queue_depth: float, config: ScalerConfig) -> int:
        """Calculate replicas from queue metric"""
        
        # Scale based on queue depth
        replicas_per_message = queue_depth / config.target_value if config.target_value > 0 else 1
        target = int(replicas_per_message) + 1
        
        return target
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Service Mesh & Policy

### Network Policy Manager
```python
class NetworkPolicy:
    """Kubernetes network policy"""
    
    def __init__(self, name: str, namespace: str):
        self.name = name
        self.namespace = namespace
        self.pod_selector: Dict = {}
        self.ingress_rules: List[Dict] = []
        self.egress_rules: List[Dict] = []

class NetworkPolicyController:
    """Manage network policies"""
    
    def __init__(self):
        self.policies: Dict[str, NetworkPolicy] = {}
    
    def create_policy(self, policy: NetworkPolicy) -> bool:
        """Create network policy"""
        
        policy_key = f"{policy.namespace}/{policy.name}"
        self.policies[policy_key] = policy
        
        return True
    
    def add_ingress_rule(self, policy_name: str, namespace: str, 
                        from_pods: Dict, ports: List[Dict]) -> bool:
        """Add ingress rule"""
        
        policy_key = f"{namespace}/{policy_name}"
        
        if policy_key not in self.policies:
            return False
        
        rule = {
            'from': [{'podSelector': from_pods}],
            'ports': ports
        }
        
        self.policies[policy_key].ingress_rules.append(rule)
        
        return True
    
    def add_egress_rule(self, policy_name: str, namespace: str, 
                       to_pods: Dict, ports: List[Dict]) -> bool:
        """Add egress rule"""
        
        policy_key = f"{namespace}/{policy_name}"
        
        if policy_key not in self.policies:
            return False
        
        rule = {
            'to': [{'podSelector': to_pods}],
            'ports': ports
        }
        
        self.policies[policy_key].egress_rules.append(rule)
        
        return True
    
    def evaluate_connection(self, source_pod: Dict, dest_pod: Dict, port: int) -> bool:
        """Evaluate if connection allowed"""
        
        # Check all policies
        for policy in self.policies.values():
            # Check if destination matches policy selector
            if not self._matches_selector(dest_pod, policy.pod_selector):
                continue
            
            # Check ingress rules
            for rule in policy.ingress_rules:
                if self._matches_rule(source_pod, dest_pod, port, rule):
                    return True
        
        return False
    
    def _matches_selector(self, pod: Dict, selector: Dict) -> bool:
        """Check if pod matches selector"""
        
        for key, value in selector.items():
            if pod.get(key) != value:
                return False
        
        return True
    
    def _matches_rule(self, source_pod: Dict, dest_pod: Dict, port: int, rule: Dict) -> bool:
        """Check if connection matches rule"""
        
        # Check source
        for from_rule in rule.get('from', []):
            if not self._matches_selector(source_pod, from_rule['podSelector']):
                continue
            
            # Check port
            for port_rule in rule.get('ports', []):
                if port_rule['port'] == port:
                    return True
        
        return False
```

### Resource Quota Management
```python
class ResourceQuota:
    """Kubernetes resource quota"""
    
    def __init__(self, namespace: str):
        self.namespace = namespace
        self.limits: Dict[str, float] = {
            'cpu': 100.0,
            'memory': 1000.0,
            'pods': 100
        }
        self.usage: Dict[str, float] = {
            'cpu': 0.0,
            'memory': 0.0,
            'pods': 0
        }

class QuotaController:
    """Manage resource quotas"""
    
    def __init__(self):
        self.quotas: Dict[str, ResourceQuota] = {}
    
    def create_quota(self, namespace: str, limits: Dict) -> bool:
        """Create resource quota"""
        
        quota = ResourceQuota(namespace)
        quota.limits = limits
        
        self.quotas[namespace] = quota
        
        return True
    
    def check_quota(self, namespace: str, resource_type: str, amount: float) -> bool:
        """Check if resource allocation allowed"""
        
        if namespace not in self.quotas:
            return True
        
        quota = self.quotas[namespace]
        
        if resource_type not in quota.limits:
            return True
        
        available = quota.limits[resource_type] - quota.usage[resource_type]
        
        return amount <= available
    
    def allocate_resource(self, namespace: str, resource_type: str, amount: float) -> bool:
        """Allocate resource"""
        
        if not self.check_quota(namespace, resource_type, amount):
            return False
        
        self.quotas[namespace].usage[resource_type] += amount
        
        return True
    
    def release_resource(self, namespace: str, resource_type: str, amount: float) -> bool:
        """Release resource"""
        
        if namespace not in self.quotas:
            return False
        
        self.quotas[namespace].usage[resource_type] -= amount
        
        return True
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Kubernetes Advanced Checklist

✅ **CRDs & Operators**
- [ ] CRDs defined
- [ ] Operator implemented
- [ ] Reconciliation logic
- [ ] Status updates
- [ ] Event handling

✅ **KEDA Autoscaling**
- [ ] Scalers configured
- [ ] Min/max replicas
- [ ] Metrics collection
- [ ] Scaling policies
- [ ] Testing completed

✅ **Network Policies**
- [ ] Policies defined
- [ ] Ingress rules
- [ ] Egress rules
- [ ] Policy testing
- [ ] Monitoring

✅ **Resource Management**
- [ ] Quotas configured
- [ ] Limits set
- [ ] Usage tracking
- [ ] Alerts configured
- [ ] Enforcement verified

✅ **Operational**
- [ ] Documentation
- [ ] Team training
- [ ] Monitoring dashboard
- [ ] Troubleshooting guides
- [ ] Disaster recovery
</VSCode.Cell>
```